In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

BRONZE_DB = "petroleum_bronze"
SILVER_DB = "petroleum_silver"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")
print("Silver database ready.")


In [0]:
df_raw_gasoline=spark.table(f"{BRONZE_DB}.raw_gasoline_price")

In [0]:
df_raw_gasoline.printSchema()

In [0]:
def clean_series(table_name: str, value_col_alias: str):
    """
    Read a bronze table, cast types, drop nulls, return clean DataFrame.
    No date filtering — full history is preserved.
    """
    df = spark.table(f"{BRONZE_DB}.{table_name}")

    df_clean = (
        df
        .withColumn("date",         F.to_date(F.col("date"), "yyyy-MM-dd"))
        .withColumn("value_numeric", F.col("value").try_cast("double"))
        .filter(F.col("value_numeric").isNotNull())
        .filter(F.col("date").isNotNull())
        .select(
            F.col("date"),
            F.col("value_numeric").alias(value_col_alias),
            F.col("series_id"),
            F.col("series_label"),
            F.col("ingested_at")
        )
        .dropDuplicates(["date"])
        .orderBy("date")
    )
    print(f"  {table_name}: {df_clean.count()} clean rows")
    return df_clean

df_wti        = clean_series("raw_wti_price",        "wti_price")
df_brent      = clean_series("raw_brent_price",      "brent_price")
df_production = clean_series("raw_us_production",    "us_production_mbpd")
df_stocks     = clean_series("raw_crude_stocks",     "crude_stocks_mb")
df_supplied   = clean_series("raw_product_supplied", "product_supplied_mbpd")
df_gasoline   = clean_series("raw_gasoline_price",   "gasoline_price_gal")
df_diesel     = clean_series("raw_diesel_price",     "diesel_price_gal")
print("\nAll series cleaned.")


In [0]:
# WTI is the spine — it has the longest continuous history
# All other series left-joined so gaps don't drop WTI rows
df_silver = (
    df_wti.select("date", "wti_price")
    .join(df_brent.select("date", "brent_price"),              on="date", how="left")
    .join(df_production.select("date", "us_production_mbpd"),  on="date", how="left")
    .join(df_stocks.select("date", "crude_stocks_mb"),         on="date", how="left")
    .join(df_supplied.select("date", "product_supplied_mbpd"), on="date", how="left")
    .join(df_gasoline.select("date", "gasoline_price_gal"),    on="date", how="left")
    .join(df_diesel.select("date", "diesel_price_gal"),        on="date", how="left")
    .orderBy("date")
)
print(f"Joined silver table: {df_silver.count()} rows")
df_silver.show(3)


In [0]:
# -----_Adding Rolling calculations, flags and era lables------------
w_date    = Window.orderBy("date")
w_4wk     = Window.orderBy("date").rowsBetween(-3, 0)
w_52wk    = Window.orderBy("date").rowsBetween(-52, 0)

df_silver_enriched = (
    df_silver

    # ── WTI calculations ──────────────────────────────────────────
    .withColumn("wti_prev_week",   F.lag("wti_price", 1).over(w_date))
    .withColumn("wti_wow_pct",     F.round(
        (F.col("wti_price") - F.col("wti_prev_week")) / F.col("wti_prev_week") * 100, 2))
    .withColumn("wti_4wk_avg",     F.round(F.avg("wti_price").over(w_4wk), 2))

    # ── Brent calculations ────────────────────────────────────────
    .withColumn("brent_prev_week", F.lag("brent_price", 1).over(w_date))
    .withColumn("brent_wow_pct",   F.round(
        (F.col("brent_price") - F.col("brent_prev_week")) / F.col("brent_prev_week") * 100, 2))
    .withColumn("wti_brent_spread",F.round(F.col("wti_price") - F.col("brent_price"), 2))

    # ── Stocks calculations ───────────────────────────────────────
    .withColumn("stocks_prev_week",F.lag("crude_stocks_mb", 1).over(w_date))
    .withColumn("stocks_wow_chg",  F.round(F.col("crude_stocks_mb") - F.col("stocks_prev_week"), 2))

    # ── Gasoline rolling avg (52-week) ────────────────────────────
    .withColumn("gas_52wk_avg",    F.round(F.avg("gasoline_price_gal").over(w_52wk), 3))

    # ── Conflict flag ─────────────────────────────────────────────
    .withColumn("conflict_flag",
        F.when(F.col("date") >= F.lit("2026-01-05").cast("date"), 1).otherwise(0))

    # ── ERA LABEL (new in v2) ─────────────────────────────────────
    # Labels each row with its historical context for Tableau filtering
    .withColumn("era",
        F.when(F.col("date") < F.lit("1986-01-01").cast("date"), "1973-1985 Oil Shock Era")
        .when(F.col("date") < F.lit("1991-08-01").cast("date"), "1986-1991 Glut & Recovery")
        .when(F.col("date") < F.lit("1991-12-31").cast("date"), "1991 Gulf War")
        .when(F.col("date") < F.lit("2008-01-01").cast("date"), "1992-2007 Stable Growth")
        .when(F.col("date") < F.lit("2010-01-01").cast("date"), "2008 Financial Crisis")
        .when(F.col("date") < F.lit("2020-01-01").cast("date"), "2010-2019 Shale Boom")
        .when(F.col("date") < F.lit("2022-01-01").cast("date"), "2020 COVID Collapse")
        .when(F.col("date") < F.lit("2026-01-05").cast("date"), "2022-2025 Post-COVID")
        .otherwise("2026 Iran Conflict"))

    # ── Date parts for Tableau ────────────────────────────────────
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("week",  F.weekofyear("date"))

    # ── Drop intermediate columns ─────────────────────────────────
    .drop("wti_prev_week", "brent_prev_week", "stocks_prev_week")
)

print(f"Enriched rows: {df_silver_enriched.count()}")
df_silver_enriched.select("date", "era", "wti_price", "conflict_flag") \
    .orderBy("date", ascending=False).show(10)


In [0]:
df_silver_enriched.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{SILVER_DB}.petroleum_weekly")

df_check = spark.table(f"{SILVER_DB}.petroleum_weekly")
print("=== SILVER VALIDATION ===")
print(f"Total rows:          {df_check.count()}")
print(f"Date range:          {df_check.agg(F.min('date'), F.max('date')).collect()[0]}")
print(f"Null WTI prices:     {df_check.filter(F.col('wti_price').isNull()).count()}")
print(f"Conflict rows:       {df_check.filter(F.col('conflict_flag')==1).count()}")

# Show era distribution
print("\n--- Row count by era ---")
df_check.groupBy("era").count().orderBy("era").show(truncate=False)
